# Transpilation

This notebook covers transpiling circuits for hardware targets, comparing
optimization levels, and circuit composition utilities.

In [1]:
import (
	"context"
	"fmt"

	"github.com/janpfeifer/gonb/gonbui"
	"github.com/splch/goqu/circuit/builder"
	"github.com/splch/goqu/circuit/draw"
	"github.com/splch/goqu/circuit/ir"
	"github.com/splch/goqu/transpile/pipeline"
	"github.com/splch/goqu/transpile/target"
)

In [2]:
var ctx = context.Background()

func buildToffoli() *ir.Circuit {
	c, _ := builder.New("toffoli", 3).H(2).CCX(0, 1, 2).H(2).Build()
	return c
}

## Transpiling for Hardware

Build a circuit with a Toffoli gate (CCX) that needs decomposition for
hardware targets that only support 1- and 2-qubit basis gates.

In [3]:
%%
gonbui.DisplaySvg(draw.SVG(buildToffoli()))

q0 
 
 q1 
 
 q2 
 
 H 
 
 
 
 
 H

In [4]:
%%
c := buildToffoli()
transpiled, _ := pipeline.Run(ctx, c, target.IBMBrisbane, pipeline.LevelFull)
fmt.Printf("Before: depth=%d gates=%d 2q=%d\n",
	c.Stats().Depth, c.Stats().GateCount, c.Stats().TwoQubitGates)
fmt.Printf("After:  depth=%d gates=%d 2q=%d\n",
	transpiled.Stats().Depth, transpiled.Stats().GateCount, transpiled.Stats().TwoQubitGates)

Before: depth=3 gates=3 2q=1
After:  depth=17 gates=22 2q=6


## Comparing Optimization Levels

Goqu provides four optimization levels:
- **LevelNone (0):** Decompose only
- **LevelBasic (1):** + cancel + merge
- **LevelFull (2):** + commute + parallelize
- **LevelParallel (3):** Multi-strategy, pick best

In [5]:
%%
levels := []struct {
	name  string
	level pipeline.Level
}{
	{"None", pipeline.LevelNone},
	{"Basic", pipeline.LevelBasic},
	{"Full", pipeline.LevelFull},
	{"Parallel", pipeline.LevelParallel},
}

for _, l := range levels {
	out, _ := pipeline.Run(ctx, buildToffoli(), target.IBMBrisbane, l.level)
	s := out.Stats()
	fmt.Printf("%-10s depth=%-3d gates=%-3d 2q=%-3d\n", l.name, s.Depth, s.GateCount, s.TwoQubitGates)
}

None       depth=20  gates=25  2q=6  
Basic      depth=17  gates=22  2q=6  
Full       depth=17  gates=22  2q=6  
Parallel   depth=17  gates=22  2q=6  


## Before / After SVG

Display the original and transpiled circuits side by side using inline HTML.

In [6]:
%%
c := buildToffoli()
transpiled, _ := pipeline.Run(ctx, c, target.IBMBrisbane, pipeline.LevelFull)
html := fmt.Sprintf(
	`<div style="display:flex;gap:2em"><div><h4>Original</h4>%s</div><div><h4>Transpiled (LevelFull)</h4>%s</div></div>`,
	draw.SVG(c), draw.SVG(transpiled),
)
gonbui.DisplayHTML(html)

Original 
 
 q0 
 
 q1 
 
 q2 
 
 H 
 
 
 
 
 H 
 Transpiled (LevelFull) 
 
 q0 
 
 q1 
 
 q2 
 
 RZ(pi/2) 
 SX 
 RZ(pi) 
 SX 
 RZ(pi/2) 
 
 
 
 RZ(-pi/4) 
 
 
 
 RZ(pi/4) 
 
 
 
 RZ(-pi/4) 
 RZ(pi/4) 
 
 
 
 RZ(3*pi/4) 
 
 
 
 SX 
 RZ(pi/4) 
 RZ(-pi/4) 
 RZ(pi) 
 
 
 
 SX 
 RZ(pi/2)

## Circuit Composition

Goqu provides utilities to compose circuits:
- `ir.Compose()` — append one circuit after another
- `ir.Tensor()` — place circuits on disjoint qubit spaces
- `ir.Inverse()` — reverse and adjoint a circuit

In [7]:
%%
a, _ := builder.New("a", 2).H(0).CNOT(0, 1).Build()
b, _ := builder.New("b", 2).X(0).Z(1).Build()

composed, _ := ir.Compose(a, b, nil, nil)
fmt.Printf("Compose: %d qubits, depth %d\n", composed.NumQubits(), composed.Stats().Depth)
gonbui.DisplaySvg(draw.SVG(composed))

Compose: 2 qubits, depth 3


q0 
 
 q1 
 
 H 
 
 
 
 X 
 Z

In [8]:
%%
a2, _ := builder.New("a", 2).H(0).CNOT(0, 1).Build()
b2, _ := builder.New("b", 2).X(0).Z(1).Build()

tensored := ir.Tensor(a2, b2)
fmt.Printf("Tensor: %d qubits, depth %d\n", tensored.NumQubits(), tensored.Stats().Depth)
gonbui.DisplaySvg(draw.SVG(tensored))

Tensor: 4 qubits, depth 2


q0 
 
 q1 
 
 q2 
 
 q3 
 
 H 
 
 
 
 X 
 Z

In [9]:
%%
a3, _ := builder.New("a", 2).H(0).CNOT(0, 1).Build()
inv := ir.Inverse(a3)
fmt.Printf("Inverse: %d qubits, depth %d\n", inv.NumQubits(), inv.Stats().Depth)
gonbui.DisplaySvg(draw.SVG(inv))

Inverse: 2 qubits, depth 2


q0 
 
 q1 
 
 
 
 
 H